# 12_feature_statistics_CHS

- 목적: 22·23 표 피처 모델 착수 전에 윈도우 피처 33열의 통계 구조(중복·상관·VIF, PCA, 그룹 비교, 분포·변환, 성능 신뢰구간)를 확인해 "입력 피처 계약 v2"(권장 열·제외 열·변환·상태 정규화)를 정한다. 각 절은 가설 → 방법 → 표 → 판정(채택/기각/보류)과 모델링 영향으로 끝난다.
- 입력: `data/processed/11_window_features.parquet` (1초 윈도우 기본, PCA 스모크만 1·2·3초). `segments.csv`·원시 데이터 로드 없음
- 출력: `figures/12_feature_statistics_CHS/`, `results/12_feature_statistics_CHS/` (`duplicate_columns.csv`, `corr_matrix.csv`, `corr_clusters.csv`, `corr_by_state_diff.csv`, `vif_table.csv`, `reduced_feature_set.csv`, `vif_table_reduced.csv`, `pca_explained.csv`, `pca_loadings.csv`, `pca_recon_smoke.csv`, `group_test_normal_vs_outlier.csv`, `group_test_by_state.csv`, `distribution_summary.csv`, `transform_effect.csv`, `auc_bootstrap_21.csv`, `auc_diff_21.csv`)
- 담당: CHS
- 심사 대응: 2번(입력 계약·PCA 재구성 후보·신뢰구간 규칙), 3번(정상–이상·상태 간 효과 크기), 6번(`src/stats.py` 재현, fold별 적합)

원칙: 표준화·PCA·임계·상태 경계는 **train fold 정상 윈도우**로만 적합한다. 검정 p값은 윈도우가 아니라 **세그먼트 대표값**(`stats.segment_representatives`)에서만 계산한다. 이상은 이벤트 1건(17개 평가 가능 세그먼트)이므로 정상 vs 이상 결과는 모집단 추론이 아니라 이 이벤트의 기술(記述)이다. 근거 수준은 [데이터]/[추정]/[가정]으로 표기한다.

In [1]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats as sst
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
import paths
NB = "12_feature_statistics_CHS"  # 파일명(확장자 제외)과 같게
FIG, RES = paths.nb_dirs(NB)
SEED = 42
np.random.seed(SEED)
plt.rcParams["font.family"] = "Malgun Gothic"; plt.rcParams["axes.unicode_minus"] = False
import evaluate, features, split, stats
from sklearn.cluster import KMeans
PQ = paths.DATA_PROCESSED / "11_window_features.parquet"
print(NB, FIG.relative_to(ROOT), RES.relative_to(ROOT))


12_feature_statistics_CHS figures\12_feature_statistics_CHS results\12_feature_statistics_CHS


In [2]:
W_ALL = pd.read_parquet(PQ)
W1 = W_ALL[W_ALL.win_s == 1.0].reset_index(drop=True)
FEATS = [c for c in features.feature_columns(W_ALL) if c not in features.META_COLS]
print("피처 열 수:", len(FEATS), "(33이어야 함)")
assert len(FEATS) == 33
assert not set(FEATS) & {"src", "label", "t_abs", "state", "fold", "block", "vib_grade", "cur_grade"}
print(W_ALL.groupby(["win_s", "label"]).size().unstack())
print("1초 윈도우 NaN 피처 열 수:", int(W1[FEATS].isna().any().sum()))
def group_of(c):
    if "_shape_" in c: return "shape"
    if c.startswith("cur_fit"): return "sine"
    if c in ("vib_corr01", "cur_ac1", "vib_rms_ratio"): return "rel"
    return "amp"
GCOL = {"amp": "#1f77b4", "shape": "#2ca02c", "sine": "#d62728", "rel": "#9467bd"}
TR0 = W1[(W1.fold != 0) & (W1.label == 0)]        # fold 0 train 정상 윈도우
print("fold 0 train 정상 윈도우:", len(TR0), "세그먼트:", TR0.seg_uid.nunique())

피처 열 수: 33 (33이어야 함)
label     0   1
win_s          
1.0    3264  96
2.0    2254  63
3.0    1400  38
1초 윈도우 NaN 피처 열 수: 0
fold 0 train 정상 윈도우: 2605 세그먼트: 421


## 1. E1 — 상관·중복·VIF

- 가설: 진폭군 rms·peak·p2p는 거의 같은 정보이고, `cur_ac1`은 사인 잔차와, `vib_corr01`은 `vib_rms_ratio`와 겹친다. 형상(z-score) 첨도·왜도는 스케일 불변이라 진폭군 `kurt`·`skew`와 **완전 중복**일 것이다. Mahalanobis(23)는 공분산이 특이해 그대로 못 쓴다.
- 방법: (a) 33열 쌍별 max|a−b| < 1e-9 완전 중복 검사(1초 윈도우 전체) → 중복 제거. (b) 남은 열로 fold 0 train 정상 윈도우 Spearman 상관·|ρ|≥0.9 군집, 상태(고부하 state 0 / 저부하 state 1)별 상관 차이. (c) VIF·조건수. (d) 축소 입력 집합(|ρ|≥0.95 군집은 대표 1열).

In [3]:
# (a) 완전 중복 검사
rows = []
for i, a in enumerate(FEATS):
    for b in FEATS[i + 1:]:
        d = (W1[a] - W1[b]).abs().max()
        rows.append({"col_a": a, "col_b": b, "max_abs_diff": float(d)})
PAIRS = pd.DataFrame(rows)
DUP = PAIRS[PAIRS.max_abs_diff < 1e-9].sort_values("max_abs_diff").reset_index(drop=True)
DUP.to_csv(RES / "duplicate_columns.csv", index=False)
print("검사 쌍:", len(PAIRS), "/ 완전 중복 쌍(<1e-9):", len(DUP))
print(DUP.to_string(index=False))
print("중복 쌍 max|diff| 최대: %.2e / 비중복 쌍 중 최소 max|diff|: %.3e" % (DUP.max_abs_diff.max(), PAIRS[PAIRS.max_abs_diff >= 1e-9].max_abs_diff.min()))
DROP_DUP = {}                      # 제거 열(형상 열) -> 남기는 열(진폭 열)
for r in DUP.itertuples():
    if "_shape_" in r.col_b: DROP_DUP[r.col_b] = r.col_a
    else: DROP_DUP[r.col_a] = r.col_b
F27 = [c for c in FEATS if c not in DROP_DUP]
print("중복 제거 후 열 수:", len(F27))

검사 쌍: 528 / 완전 중복 쌍(<1e-9): 6
             col_a                    col_b  max_abs_diff
AI0_Vibration_skew AI0_Vibration_shape_skew  2.386980e-15
  AI2_Current_skew   AI2_Current_shape_skew  3.719247e-15
  AI2_Current_kurt   AI2_Current_shape_kurt  3.996803e-15
AI1_Vibration_skew AI1_Vibration_shape_skew  3.996803e-15
AI0_Vibration_kurt AI0_Vibration_shape_kurt  4.440892e-15
AI1_Vibration_kurt AI1_Vibration_shape_kurt  5.329071e-15
중복 쌍 max|diff| 최대: 5.33e-15 / 비중복 쌍 중 최소 max|diff|: 4.249e-01
중복 제거 후 열 수: 27


In [4]:
# (b) fold 0 train 정상 윈도우 Spearman 상관 · |rho|>=0.9 군집
CORR, CL = stats.corr_clusters(TR0, F27, method="spearman", thr=0.9)
CORR.to_csv(RES / "corr_matrix.csv"); CL.to_csv(RES / "corr_clusters.csv", index=False)
print("0.9 군집 수: %d (다중 열 군집 %d개, 단독 열 %d개)" % (CL.cluster.nunique(), (CL.groupby("cluster").size() > 1).sum(), (CL.groupby("cluster").size() == 1).sum()))
print(CL[CL.cluster_size > 1].sort_values(["cluster", "feature"]).to_string(index=False))
order = CL.sort_values(["cluster", "feature"]).feature.tolist()
fig, ax = plt.subplots(figsize=(11, 9.5))
im = ax.imshow(CORR.loc[order, order].to_numpy(), vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=90, fontsize=7)
ax.set_yticks(range(len(order))); ax.set_yticklabels(order, fontsize=7)
cs = CL.set_index("feature").loc[order].cluster.to_numpy()
for k in range(1, len(cs)):
    if cs[k] != cs[k - 1]:
        ax.axhline(k - 0.5, color="k", lw=0.5); ax.axvline(k - 0.5, color="k", lw=0.5)
fig.colorbar(im, ax=ax, shrink=0.7, label="Spearman rho")
ax.set_title("fold 0 train 정상 1초 윈도우 상관행렬 (27열, |rho|>=0.9 군집 순서)")
paths.save_fig(FIG, "corr_heatmap.png")
# 상태별 상관 차이
# parquet state 규약: 0 = 고부하(전류 AC RMS 큼), 1 = 저부하 — 아래에서 전류 RMS 중앙값으로 확인
print("state별 전류 AC RMS 중앙값:", TR0.groupby("state")["AI2_Current_rms"].median().round(1).to_dict(), "/ state 비율:", TR0.state.value_counts(normalize=True).round(4).to_dict())
hi, lo = TR0[TR0.state == 0], TR0[TR0.state == 1]      # hi = 고부하, lo = 저부하
C0, C1 = hi[F27].corr(method="spearman"), lo[F27].corr(method="spearman")
rows = []
for i, a in enumerate(F27):
    for b in F27[i + 1:]:
        rows.append({"feature_a": a, "feature_b": b, "rho_state0_high": C0.loc[a, b], "rho_state1_low": C1.loc[a, b],
                     "abs_diff": abs(C0.loc[a, b] - C1.loc[a, b])})
SD = pd.DataFrame(rows).sort_values("abs_diff", ascending=False).head(10).reset_index(drop=True)
SD.to_csv(RES / "corr_by_state_diff.csv", index=False)
print("상태별 윈도우 수: 고부하(state0) %d / 저부하(state1) %d" % (len(hi), len(lo)))
print(SD.round(3).to_string(index=False))

0.9 군집 수: 19 (다중 열 군집 6개, 단독 열 13개)
                  feature  cluster  cluster_size     representative  is_representative
        AI0_Vibration_p2p        1             3  AI0_Vibration_p2p               True
       AI0_Vibration_peak        1             3  AI0_Vibration_p2p              False
        AI0_Vibration_rms        1             3  AI0_Vibration_p2p              False
         AI2_Current_peak        2             2    AI2_Current_rms              False
          AI2_Current_rms        2             2    AI2_Current_rms               True
        AI1_Vibration_p2p        4             3  AI1_Vibration_p2p               True
       AI1_Vibration_peak        4             3  AI1_Vibration_p2p              False
        AI1_Vibration_rms        4             3  AI1_Vibration_p2p              False
       AI1_Vibration_kurt        8             2 AI1_Vibration_kurt               True
AI1_Vibration_shape_crest        8             2 AI1_Vibration_kurt              False
       

saved figures\12_feature_statistics_CHS\corr_heatmap.png
state별 전류 AC RMS 중앙값: {0.0: 156.3, 1.0: 86.4} / state 비율: {1.0: 0.5674, 0.0: 0.4326}
상태별 윈도우 수: 고부하(state0) 1127 / 저부하(state1) 1478
         feature_a          feature_b  rho_state0_high  rho_state1_low  abs_diff
  AI2_Current_peak  cur_fit_resid_rms           -0.363           0.363     0.726
     cur_fit_f_dev         vib_corr01           -0.307           0.400     0.707
 AI1_Vibration_p2p AI1_Vibration_skew           -0.447           0.237     0.684
 AI1_Vibration_rms AI1_Vibration_skew           -0.462           0.209     0.670
   AI2_Current_rms  cur_fit_resid_rms           -0.371           0.299     0.670
AI1_Vibration_skew      vib_rms_ratio            0.482          -0.145     0.627
        cur_fit_r2      cur_fit_f_dev           -0.464           0.151     0.615
AI1_Vibration_skew         cur_fit_r2           -0.457           0.151     0.607
AI1_Vibration_peak AI1_Vibration_skew           -0.306           0.290     0.596
 

In [5]:
# (c) VIF · 조건수
VIF = stats.vif_table(TR0, F27)
VIF["group"] = VIF.feature.map(group_of)
VIF.to_csv(RES / "vif_table.csv", index=False)
COND = float(VIF.condition_number.iloc[0])
hi_vif = VIF[VIF.vif > 10].sort_values("vif", ascending=False)
print("조건수(표준화 27열): %.3e / VIF>10 열 %d개 (inf %d개)" % (COND, len(hi_vif), int(np.isinf(VIF.vif).sum())))
print(hi_vif[["feature", "vif"]].round(1).to_string(index=False))
print("참고: 중복 제거 전 33열 조건수 %.3e (VIF inf %d개)" % (stats.vif_table(TR0, FEATS).condition_number.iloc[0], int(np.isinf(stats.vif_table(TR0, FEATS).vif).sum())))

조건수(표준화 27열): 9.918e+01 / VIF>10 열 14개 (inf 0개)
                feature   vif
        AI2_Current_rms 488.4
       AI2_Current_peak 479.2
      AI1_Vibration_rms 139.2
     AI1_Vibration_peak 138.9
      AI1_Vibration_p2p 106.8
     AI0_Vibration_peak  67.2
      AI0_Vibration_rms  63.4
        AI2_Current_p2p  21.0
             cur_fit_r2  20.5
      AI2_Current_crest  17.7
AI2_Current_shape_crest  16.7
       AI2_Current_kurt  16.2
      AI0_Vibration_p2p  15.3
    AI0_Vibration_crest  10.8
참고: 중복 제거 전 33열 조건수 9.715e+15 (VIF inf 12개)


In [6]:
# (d) 축소 입력 집합: 중복 제거 -> |rho|>=0.95 군집은 대표만
_, CL95 = stats.corr_clusters(TR0, F27, method="spearman", thr=0.95)
rep95 = CL95.set_index("feature")
rows = []
for c in FEATS:
    if c in DROP_DUP:
        reason = "dropped_duplicate:%s" % DROP_DUP[c]
    elif rep95.loc[c, "is_representative"]:
        reason = "kept"
    else:
        reason = "dropped_cluster:%s" % rep95.loc[c, "representative"]
    rows.append({"feature": c, "group": group_of(c), "reason": reason})
RED = pd.DataFrame(rows)
RED.to_csv(RES / "reduced_feature_set.csv", index=False)
FRED = RED[RED.reason == "kept"].feature.tolist()
VIF_R = stats.vif_table(TR0, FRED); VIF_R["group"] = VIF_R.feature.map(group_of)
VIF_R.to_csv(RES / "vif_table_reduced.csv", index=False)
COND_R = float(VIF_R.condition_number.iloc[0])
print("0.95 군집 수 %d / 축소 집합 열 수 %d (33 -> 27 -> %d)" % (CL95.cluster.nunique(), len(FRED), len(FRED)))
print(RED[RED.reason != "kept"].to_string(index=False))
print("축소 집합 열:", FRED)
print(RED[RED.reason == "kept"].group.value_counts().to_dict())
print("조건수 전체27열 %.3e -> 축소 %.3e / VIF>10 열: 27열 %d개 -> 축소 %d개 (최대 VIF %.1f)" % (
    COND, COND_R, len(hi_vif), int((VIF_R.vif > 10).sum()), VIF_R.vif.replace(np.inf, np.nan).max()))
print(VIF_R.sort_values("vif", ascending=False).head(8).round(2)[["feature", "vif"]].to_string(index=False))

0.95 군집 수 23 / 축소 집합 열 수 23 (33 -> 27 -> 23)
                 feature group                               reason
AI0_Vibration_shape_kurt shape dropped_duplicate:AI0_Vibration_kurt
AI0_Vibration_shape_skew shape dropped_duplicate:AI0_Vibration_skew
       AI1_Vibration_rms   amp    dropped_cluster:AI1_Vibration_p2p
      AI1_Vibration_peak   amp    dropped_cluster:AI1_Vibration_p2p
AI1_Vibration_shape_kurt shape dropped_duplicate:AI1_Vibration_kurt
AI1_Vibration_shape_skew shape dropped_duplicate:AI1_Vibration_skew
        AI2_Current_peak   amp      dropped_cluster:AI2_Current_rms
  AI2_Current_shape_kurt shape   dropped_duplicate:AI2_Current_kurt
 AI2_Current_shape_crest shape     dropped_cluster:AI2_Current_kurt
  AI2_Current_shape_skew shape   dropped_duplicate:AI2_Current_skew
축소 집합 열: ['AI0_Vibration_rms', 'AI0_Vibration_peak', 'AI0_Vibration_p2p', 'AI0_Vibration_kurt', 'AI0_Vibration_crest', 'AI0_Vibration_skew', 'AI0_Vibration_shape_crest', 'AI1_Vibration_p2p', 'AI1_Vibration_k

### E1 판정 (근거 수준 [데이터], fold 0 train 정상 1초 윈도우 2,605개·421 세그먼트)

| 가설 | 판정 | 수치 |
|---|---|---|
| 형상 kurt·skew는 진폭 kurt·skew와 완전 중복 | **채택** | 528쌍 중 6쌍이 max\|diff\| ≤ 5.33e-15 (다음으로 가까운 쌍은 0.425). 제거 대상: AI0·AI1·AI2 `*_shape_kurt`·`*_shape_skew` → 33열 → **27열** |
| rms·peak·p2p는 같은 정보 | **부분 채택** | 0.9 군집 19개(다중 열 6개): {AI0 rms·peak·p2p}, {AI1 rms·peak·p2p}, {AI2 rms·peak}, 그리고 shape_crest–kurt 3쌍(AI0·AI1·AI2). p2p는 AI2에서 rms와 묶이지 않음 |
| `cur_ac1`은 사인 잔차와, `vib_corr01`은 `vib_rms_ratio`와 겹침 | **기각** | 두 쌍 모두 0.9 군집에 묶이지 않음(sine·rel 6열은 단독 군집) |
| 공분산이 특이해 Mahalanobis 불가 | **33열에서만 채택** | 33열 조건수 9.7e15(VIF inf 12개) → 중복 제거 후 27열 **99.2**, 축소 23열 **30.8**. 계획서 기준(≥1e6)은 중복 제거 전에만 해당 |

- VIF>10: 27열에서 14개(최대 `AI2_Current_rms` 488.4, `AI2_Current_peak` 479.2, `AI1_Vibration_rms` 139.2). 축소 집합(23열)에서는 8개(최대 `AI0_Vibration_peak` 66.9, `AI0_Vibration_rms` 61.4)로 줄지만 AI0 rms·peak·p2p는 0.95 군집이 아니라 남는다.
- 축소 입력 집합 23열 = 진폭 15 + 형상 2(`AI0/AI1_shape_crest`) + 사인 3 + 관계 3. 제거 10열 = 중복 6 + 군집 4(`AI1_Vibration_rms`·`peak`→p2p, `AI2_Current_peak`→rms, `AI2_Current_shape_crest`→kurt).
- 상태 조건부 상관: 가장 크게 바뀐 쌍은 `AI2_Current_peak`–`cur_fit_resid_rms`(고부하 −0.363 / 저부하 +0.363, 차이 0.726)로 **부호가 뒤집힌다**(상위 10쌍 차이 0.577~0.726). state 규약: parquet `state` 0 = 고부하(전류 AC RMS 중앙값 156.3, 정상 윈도우 43.3%), 1 = 저부하(86.4, 56.7%).
- **모델링 영향**: (1) 22·23 입력은 33열이 아니라 중복 제거 27열 이하를 쓴다(33열 그대로면 공분산 특이). (2) 23 Mahalanobis는 27열 또는 축소 23열에서 바로 적합 가능(조건수 99/31, PCA 공간 강제 아님), 다만 공선 진폭군은 공분산 역행렬을 불안정하게 하므로 축소 집합 또는 수축(shrinkage) 공분산 권장. (3) 상관 구조가 상태별로 다르므로 상태별 공분산·GMM 성분(2) 변형을 후보에 둔다. 판정: **채택**(중복 제거·축소 집합), **보류**(상태별 공분산 효과는 23에서 검증).

## 2. E2 — PCA와 재구성 오차 스모크

- 가설: 정상 27열 변동은 소수 축(부하 축 + 진동 형상 축)으로 설명되고, 이상은 정상 주성분 공간 바깥(재구성 오차)에 있다.
- 방법: 27열(중복 제거) 표준화 PCA를 fold별 train 정상 윈도우로 적합(group_kfold_seg 5 fold, time_block 1~4). 재구성 오차(표준화 공간 행별 RMSE, 성분 수 = 누적 95% 도달 수)를 점수로 21과 동일한 평가 루프(임계 = train 정상 q99)로 평가한다. 대표 적재량·산점도는 fold 0.

In [7]:
PCA_COLS = F27
EXTRA = ["win_s", "thr", "n_out_seg", "n_detected", "n_components"]

def pca_score(w, tr_normal_mask):
    """train 정상으로 표준화+PCA 적합(95% 성분 수) -> 전체 행 재구성 오차와 정보. 결측은 train 정상 중앙값으로 대체."""
    trn = w.loc[tr_normal_mask, PCA_COLS]
    full = stats.pca_fit_report(trn, PCA_COLS)
    k95 = full["n_for"][0.95]
    rep = stats.pca_fit_report(trn, PCA_COLS, n_components=k95)
    x = w[PCA_COLS].fillna(trn.median())
    return stats.pca_recon_error(rep, x), {"full": full, "rep": rep, "k90": full["n_for"][0.9], "k95": k95, "x": x}

def eval_one(w, s, tr_mask, te_idx, split_name, fold_id, win_s, k95):
    thr = evaluate.threshold_from_normal(s[tr_mask & (w.label == 0).to_numpy()])
    te = w.loc[te_idx].assign(_s=s[te_idx]); n, o = te[te.label == 0], te[te.label == 1]
    dl = evaluate.detection_delay_s(o._s, o.seg_uid, o.t_start, thr, win_s=win_s)
    r = evaluate.to_metrics_row("pca_recon", split_name, fold_id, evaluate.auc(te.label, te._s),
                                evaluate.fpr_sample(n._s, thr), evaluate.fpr_segment(n._s, n.seg_uid, thr),
                                dl["delay_s"].median())
    r.update({"win_s": win_s, "thr": thr, "n_out_seg": len(dl), "n_detected": int(dl["detected"].sum()), "n_components": k95})
    return r

rows, EXPL, FOLD0 = [], [], {}
for win_s in (1.0, 2.0, 3.0):
    w = W_ALL[W_ALL.win_s == win_s].reset_index(drop=True)
    lab0 = (w.label == 0).to_numpy()
    for k in range(5):
        tr = (w.fold != k).to_numpy(); te = np.flatnonzero(w.fold == k)
        s, info = pca_score(w, tr & lab0)
        rows.append(eval_one(w, s, tr, te, "group_kfold_seg", k, win_s, info["k95"]))
        if win_s == 1.0:
            e = info["full"]["explained"].assign(fold=k, n90=info["k90"], n95=info["k95"])
            EXPL.append(e[["fold", "component", "explained_ratio", "cumulative", "n90", "n95"]])
            if k == 0: FOLD0 = {"info": info, "te": te, "w": w}
    for k in range(1, 5):
        tr = (w.block < k).fillna(False).to_numpy(bool)
        te = split.append_outliers(np.flatnonzero((w.block == k).fillna(False).to_numpy(bool)), w)
        s, info = pca_score(w, tr & lab0)
        rows.append(eval_one(w, s, tr, te, "time_block", k, win_s, info["k95"]))
PM = pd.DataFrame(rows)
mean_rows = []
for (sp, m, ws), g_ in PM.groupby(["split", "model", "win_s"], sort=False):
    r = g_.drop(columns=["model", "split", "fold"]).mean(numeric_only=True).to_dict()
    r.update({"model": m, "split": sp, "fold": "mean"}); mean_rows.append(r)
PM = pd.concat([PM, pd.DataFrame(mean_rows)], ignore_index=True); PM["fold"] = PM["fold"].astype(str)
PM = PM[evaluate.METRIC_COLS + EXTRA]
PM.to_csv(RES / "pca_recon_smoke.csv", index=False)
EX = pd.concat(EXPL, ignore_index=True); EX.to_csv(RES / "pca_explained.csv", index=False)
print(PM[PM.fold == "mean"].round(4).to_string(index=False))

    model           split fold  auc  fpr_sample  fpr_segment  delay_median_s  win_s    thr  n_out_seg  n_detected  n_components
pca_recon group_kfold_seg mean  1.0      0.0103       0.0550             0.9    1.0 0.4651        3.4         3.4          13.0
pca_recon      time_block mean  1.0      0.0121       0.0615             0.9    1.0 0.4843       17.0        17.0          13.0
pca_recon group_kfold_seg mean  1.0      0.0140       0.0569             1.9    2.0 0.4164        2.6         2.6          12.0
pca_recon      time_block mean  1.0      0.0246       0.0962             1.9    2.0 0.4182       13.0        13.0          12.0
pca_recon group_kfold_seg mean  1.0      0.0106       0.0165             2.9    3.0 0.4369        2.0         2.0          11.0
pca_recon      time_block mean  1.0      0.0459       0.0954             2.9    3.0 0.4263       10.0        10.0          11.0


In [8]:
# 설명 분산 · fold별 성분 수 · 적재량 · 산점도 (1초)
nn = EX.groupby("fold")[["n90", "n95"]].first()
print("fold별 90%/95% 성분 수:\n", nn.T.to_string())
e0 = EX[EX.fold == 0]
print("fold 0 PC1~5 설명 비율:", e0.explained_ratio.head(5).round(4).tolist(), "누적 PC1~3: %.4f" % e0.cumulative.iloc[2])
L0 = FOLD0["info"]["full"]["loadings"][["PC1", "PC2", "PC3"]]
rows = []
for pc in L0.columns:
    t = L0[pc].reindex(L0[pc].abs().sort_values(ascending=False).index)
    for r_, (f, val) in enumerate(t.items(), 1):
        rows.append({"pc": pc, "rank": r_, "feature": f, "loading": val, "abs_loading": abs(val), "group": group_of(f)})
LD = pd.DataFrame(rows); LD.to_csv(RES / "pca_loadings.csv", index=False)
for pc in ("PC1", "PC2", "PC3"):
    print(pc, LD[(LD.pc == pc) & (LD["rank"] <= 6)][["feature", "loading"]].round(3).values.tolist())
info = FOLD0["info"]; w0 = FOLD0["w"]; te0 = FOLD0["te"]
z = info["full"]["scaler"].transform(info["x"].iloc[te0]); P = info["full"]["pca"].transform(z)
t0 = w0.loc[te0].reset_index(drop=True)
fig, ax = plt.subplots(figsize=(8, 6))
for stt, col, nm in ((0, "#f4a261", "정상 고부하(state 0)"), (1, "#8ecae6", "정상 저부하(state 1)")):
    m_ = ((t0.label == 0) & (t0.state == stt)).to_numpy()
    ax.scatter(P[m_, 0], P[m_, 1], s=8, c=col, alpha=.6, label=nm)
m_ = (t0.label == 1).to_numpy()
ax.scatter(P[m_, 0], P[m_, 1], s=90, c="#d62728", marker="*", edgecolor="k", lw=.4, label="이상")
ax.set_xlabel("PC1 (%.1f%%)" % (100 * e0.explained_ratio.iloc[0])); ax.set_ylabel("PC2 (%.1f%%)" % (100 * e0.explained_ratio.iloc[1]))
ax.set_title("fold 0 test 1초 윈도우 PC1-PC2 (train 정상으로 적합)"); ax.legend()
paths.save_fig(FIG, "pca_scatter.png")
r1 = PM[(PM.split == "group_kfold_seg") & (PM.win_s == 1.0) & (PM.fold == "mean")].iloc[0]
tb = PM[(PM.split == "time_block") & (PM.win_s == 1.0) & (PM.fold == "mean")].iloc[0]
print("pca_recon 1초 kfold 평균: AUC %.4f / fpr_segment %.4f / 탐지 %.1f/%.1f(fold 평균) / 성분수 평균 %.1f" % (r1.auc, r1.fpr_segment, r1.n_detected, r1.n_out_seg, r1.n_components))
print("미탐 fold별(1초 kfold):", PM[(PM.split == "group_kfold_seg") & (PM.win_s == 1.0) & (PM.fold != "mean")].assign(miss=lambda d: d.n_out_seg - d.n_detected)[["fold", "miss"]].values.tolist())
print("pca_recon 1초 time_block 평균: AUC %.4f / fpr_segment %.4f / 탐지 %.1f/%.1f" % (tb.auc, tb.fpr_segment, tb.n_detected, tb.n_out_seg))
M21 = pd.read_csv(paths.RESULTS / "21_model_rule_baseline_CHS" / "metrics.csv")
m21 = M21[(M21.split == "group_kfold_seg") & (M21.win_s == 1.0) & (M21.fold.astype(str) == "mean")].set_index("model")
print("21 1초 kfold 평균 AUC:", m21.auc.round(4).to_dict())

fold별 90%/95% 성분 수:
 fold   0   1   2   3   4
n90   10  10  10  10  10
n95   13  13  13  13  13
fold 0 PC1~5 설명 비율: [0.3747, 0.0988, 0.0898, 0.0833, 0.0616] 누적 PC1~3: 0.5633
PC1 [['AI1_Vibration_rms', 0.305], ['AI2_Current_peak', 0.3], ['cur_fit_r2', 0.299], ['AI1_Vibration_p2p', 0.299], ['AI1_Vibration_peak', 0.296], ['AI2_Current_rms', 0.295]]
PC2 [['AI0_Vibration_shape_crest', 0.533], ['AI0_Vibration_kurt', 0.532], ['AI0_Vibration_crest', 0.503], ['AI2_Current_shape_crest', 0.173], ['AI2_Current_kurt', 0.169], ['AI1_Vibration_kurt', 0.168]]
PC3 [['AI2_Current_shape_crest', 0.413], ['AI2_Current_kurt', 0.407], ['AI2_Current_crest', 0.313], ['AI1_Vibration_shape_crest', 0.286], ['AI1_Vibration_kurt', 0.267], ['AI1_Vibration_skew', 0.24]]
saved figures\12_feature_statistics_CHS\pca_scatter.png
pca_recon 1초 kfold 평균: AUC 1.0000 / fpr_segment 0.0550 / 탐지 3.4/3.4(fold 평균) / 성분수 평균 13.0
미탐 fold별(1초 kfold): [['0', 0.0], ['1', 0.0], ['2', 0.0], ['3', 0.0], ['4', 0.0]]
pca_recon 1초 time_block

In [9]:
# 보조 ablation: 재구성 오차 PCA의 입력을 피처군별로 나눠 1초 kfold 평균을 비교 (sine·rel은 세그먼트 브로드캐스트 값이라 지배 여부 확인)
ABL = []
w = W1; lab0 = (w.label == 0).to_numpy()
for name, cols_ in (("all27", F27), ("amp+shape(진폭·형상 20열)", [c for c in F27 if group_of(c) in ("amp", "shape")]),
                    ("sine+rel(6열)", [c for c in F27 if group_of(c) in ("sine", "rel")])):
    PCA_COLS = cols_
    for k in range(5):
        tr = (w.fold != k).to_numpy(); te = np.flatnonzero(w.fold == k)
        s, info = pca_score(w, tr & lab0)
        r = eval_one(w, s, tr, te, "group_kfold_seg", k, 1.0, info["k95"]); r["model"] = "pca_recon:" + name; ABL.append(r)
ABL = pd.DataFrame(ABL)
AB = ABL.groupby("model")[["auc", "fpr_sample", "fpr_segment", "n_out_seg", "n_detected", "n_components"]].mean().reset_index()
AB.to_csv(RES / "pca_recon_ablation.csv", index=False)
PCA_COLS = F27
print(AB.round(4).to_string(index=False))
print("fold별 AUC:", {m: g_.auc.round(3).tolist() for m, g_ in ABL.groupby("model")})


                         model   auc  fpr_sample  fpr_segment  n_out_seg  n_detected  n_components
               pca_recon:all27 1.000      0.0103       0.0550        3.4         3.4          13.0
pca_recon:amp+shape(진폭·형상 20열) 0.994      0.0105       0.0589        3.4         3.4           9.0
        pca_recon:sine+rel(6열) 1.000      0.0130       0.0608        3.4         3.4           5.0
fold별 AUC: {'pca_recon:all27': [1.0, 1.0, 1.0, 1.0, 1.0], 'pca_recon:amp+shape(진폭·형상 20열)': [0.999, 0.994, 0.999, 0.997, 0.982], 'pca_recon:sine+rel(6열)': [1.0, 1.0, 1.0, 1.0, 1.0]}


### E2 판정 (근거 수준 [데이터], 27열·fold별 train 정상 적합)

| 가설 | 판정 | 수치 |
|---|---|---|
| 정상 변동이 소수 축으로 설명됨 | **기각** | 90% 10성분·95% **13성분**(27열 중), 5 fold 모두 동일. PC1 37.5%, PC1~3 누적 56.3% (fold 0) |
| PC1이 부하 축 | **채택** | PC1 적재량 상위: `AI1_Vibration_rms` 0.305, `AI2_Current_peak` 0.300, `cur_fit_r2` 0.299 (이어서 `AI1_Vibration_p2p`·`peak` 0.299·0.296, `AI2_Current_rms` 0.295). PC2 = AI0 진동 첨도·crest 축(0.53), PC3 = 전류 crest·kurt 축(0.41) |
| 이상이 정상 PC 공간 바깥 | **채택(단 상한 신호)** | 1초 kfold 평균 **AUC 1.0000**(fold별 모두 1.0), fpr_segment **0.0550**, fpr_sample 0.0103, 지연 중앙값 0.9초, **미탐 0**(fold별 n_detected = n_out_seg, 합 17/17). time_block 평균 AUC 1.0000·fpr_segment 0.0615·17/17. 21 rule3(0.8696)·rule1(0.7864) 대비 AUC 상위, fpr_segment는 rule3 0.0358·rule1 0.0395보다 높다 |

- 윈도우 2·3초(kfold): fpr_segment 0.0569 / 0.0165, AUC 모두 1.0000(2·3초 time_block은 fpr_segment 0.0962 / 0.0954로 오경보가 늘어남, 정상 블록 표본이 적음).
- ablation(`pca_recon_ablation.csv`, 1초 kfold): 진폭·형상 20열만 AUC **0.994**(fold별 0.982~0.999, fpr_segment 0.0589, 성분 9), 사인·관계 6열만 AUC 1.000(fpr_segment 0.0608, 성분 5). 즉 세그먼트 브로드캐스트인 사인·관계 피처만으로도 완전 분리되고, 진폭·형상만으로도 0.994다.
- 해석 주의 [가정]: 이상은 이벤트 1건이고 날짜=라벨이므로 AUC 1.0은 일반화 성능이 아니라 "이 이벤트가 정상 분포 바깥"이라는 기술이다. 21 규칙(AI0 진동 rms 중심)은 일부 세그먼트를 놓쳤지만 재구성 오차는 평가 가능한 17개를 전부 탐지했다. 전류 사인·crest·kurt 축의 기여로 보이며(3·19·20 포함 여부와 세그먼트별 기여는 31에서 확인) [추정]이다.
- **모델링 영향**: 23 후보에 "PCA 재구성"을 한 행으로 포함한다(성분 수 13, 27열 기준). 23 GMM/Mahalanobis의 PCA 차원은 90% 10 / 95% 13. PC1이 부하 축이므로 상태 조건부 정규화(E5 b)를 병행 변형으로 둔다. 판정: **채택**(23 후보 포함), 비교표는 E7 규칙(쌍 차이 CI)으로 서술.

## 3. E5 — 정상 vs 이상 · 저/고부하 그룹 비교

- 가설: 진폭군은 큰 효과, 형상군은 작은 효과, 관계군은 방향(부호)이 뒤집힌다. 고/저부하 사이에도 진폭군 차이가 커서 단일 임계는 상태에 따라 오경보율이 달라진다.
- 방법: 1초 윈도우를 세그먼트 대표값(중앙값)으로 줄인 뒤(정상 530·이상 17 세그먼트 근처) (a) 정상 vs 이상, (b) 정상만 고부하(state 0, 전류 AC RMS 큼) vs 저부하(state 1) 를 Welch t·Mann–Whitney U·Cohen d·Cliff δ, BH 보정으로 비교한다. 라벨을 쓰는 (a)의 열 선택은 **지도 정보 사용**이므로 ablation에만 쓴다.

In [10]:
REP = stats.segment_representatives(W1, F27, agg="median")
GA = stats.group_test_table(REP, F27, group_col="label", groups=(0, 1))
GA["group"] = GA.feature.map(group_of); GA.to_csv(RES / "group_test_normal_vs_outlier.csv", index=False)
GB = stats.group_test_table(REP[REP.label == 0], F27, group_col="state", groups=(0, 1))
GB["group"] = GB.feature.map(group_of); GB.to_csv(RES / "group_test_by_state.csv", index=False)
print("(a) n_a(정상 세그먼트)=%d, n_b(이상 세그먼트)=%d  -> 세그먼트 수 확인 (윈도우 수 3264/96이 아님)" % (GA.n_a.iloc[0], GA.n_b.iloc[0]))
print("(b) n_a(고부하 state0)=%d, n_b(저부하 state1)=%d" % (GB.n_a.iloc[0], GB.n_b.iloc[0]))
print("(a) BH 유의 열 %d/27, (b) BH 유의 열 %d/27" % (int(GA.significant.sum()), int(GB.significant.sum())))
cols = ["feature", "group", "median_a", "median_b", "cliff_delta", "cohen_d", "p_adj_bh", "median_b_over_q99_a"]
print("\n(a) 정상 vs 이상 (cliff_delta 절대값 순)")
print(GA.reindex(GA.cliff_delta.abs().sort_values(ascending=False).index)[cols].round(4).to_string(index=False))
print("\n(b) 고부하(state0) vs 저부하(state1) (cliff_delta 절대값 순, +는 저부하가 큼)")
print(GB.reindex(GB.cliff_delta.abs().sort_values(ascending=False).index)[cols[:-1]].round(4).to_string(index=False))
LOWA = GA[GA.cliff_delta.abs() < 0.3].feature.tolist()
BIGB = GB[GB.cliff_delta.abs() > 0.5].feature.tolist()
print("\n(a) |delta|<0.3 열 (%d): %s" % (len(LOWA), LOWA))
print("(b) 상태 간 |delta|>0.5 열 (%d): %s" % (len(BIGB), BIGB))
ordr = GA.reindex(GA.cliff_delta.abs().sort_values().index).feature.tolist()
fig, axs = plt.subplots(1, 2, figsize=(13, 8), sharey=True)
for ax, G, ttl in ((axs[0], GA, "정상 vs 이상 (b=이상)"), (axs[1], GB, "정상 고부하 vs 저부하 (b=저부하)")):
    t = G.set_index("feature").loc[ordr]
    ax.barh(ordr, t.cliff_delta, color=[GCOL[group_of(c)] for c in ordr])
    for x0 in (-0.5, -0.3, 0.3, 0.5): ax.axvline(x0, color="gray", ls=":", lw=.8)
    ax.axvline(0, color="k", lw=.8); ax.set_xlim(-1, 1); ax.set_title(ttl); ax.set_xlabel("Cliff delta")
axs[0].tick_params(axis="y", labelsize=8)
axs[1].legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in GCOL.values()], labels=list(GCOL), loc="lower right")
paths.save_fig(FIG, "effect_size_bar.png")

(a) n_a(정상 세그먼트)=530, n_b(이상 세그먼트)=17  -> 세그먼트 수 확인 (윈도우 수 3264/96이 아님)
(b) n_a(고부하 state0)=226, n_b(저부하 state1)=304
(a) BH 유의 열 21/27, (b) BH 유의 열 25/27

(a) 정상 vs 이상 (cliff_delta 절대값 순)
                  feature group  median_a  median_b  cliff_delta  cohen_d  p_adj_bh  median_b_over_q99_a
        cur_fit_resid_rms  sine    5.6586   56.0697       1.0000  17.8281    0.0000               7.6780
               cur_fit_r2  sine    0.9965    0.4507      -1.0000 -18.9021    0.0000               0.4509
            cur_fit_f_dev  sine    0.0011    0.4937       0.9984   4.9814    0.0000              10.7773
                  cur_ac1   rel    0.9001    0.4268      -0.9869  -6.9764    0.0000               0.4588
        AI2_Current_crest   amp    1.4454    1.7610       0.9436   3.8329    0.0000               0.9420
               vib_corr01   rel    0.2810   -0.3484      -0.7179  -1.7764    0.0000              -0.4736
         AI2_Current_kurt   amp   -1.1309   -0.6665       0.7074   3.2736    

saved figures\12_feature_statistics_CHS\effect_size_bar.png


### E5 판정 (근거 수준 [데이터], 세그먼트 대표값 기준 — 윈도우 p값 없음)

- n 확인: (a) 정상 **530** vs 이상 **17** 세그먼트, (b) 고부하(state 0) **226** vs 저부하(state 1) **304** 세그먼트 — 윈도우 수(3,264/96)가 아니다. BH 유의(q<0.05) (a) **21/27열**, (b) **25/27열**. (a)는 이벤트 1건의 기술이며 모집단 추론이 아니다.

| 가설 | 판정 | 수치 |
|---|---|---|
| 진폭군은 큰 효과 | **기각** | 진폭군 효과는 중간 수준: AI0 rms δ 0.573·peak 0.585·p2p 0.473, 전류 rms δ **−0.439**·p2p −0.476(이상이 정상보다 전류 AC가 **작다**). 가장 큰 효과는 사인·관계군: `cur_fit_resid_rms` δ 1.000, `cur_fit_r2` −1.000, `cur_fit_f_dev` 0.998, `cur_ac1` −0.987, `AI2_Current_crest` 0.944 |
| 형상군은 작은 효과 | **부분 채택** | 형상 3열 δ: `AI2_Current_shape_crest` 0.623, `AI1_Vibration_shape_crest` 0.587, `AI0_Vibration_shape_crest` −0.098 |
| 관계군 부호 뒤집힘 | **채택** | `vib_corr01` 정상 0.281 → 이상 −0.348 (δ −0.718), `cur_ac1` 0.900 → 0.427, `vib_rms_ratio` 0.710 → 1.370 |
| 상태 간 진폭 차이 큼 | **채택** | 상태 간 \|δ\|>0.5인 열 **16개**(AI1 rms·peak·p2p \|δ\| ≈ 0.995, 전류 rms·peak·p2p ≈ 0.99, `cur_fit_r2` −0.982, `vib_rms_ratio` 0.893, AI0 rms·peak·p2p 0.74~0.79) |

- (a) \|δ\|<0.3 열 **6개**(제외 후보, **"라벨 사용 = 지도 정보"**이므로 ablation 전용): `AI0_Vibration_kurt`(−0.033), `AI0_Vibration_crest`(−0.139), `AI0_Vibration_shape_crest`(−0.098), `AI1_Vibration_crest`(−0.009), `AI2_Current_peak`(−0.084), `AI2_Current_skew`(−0.236). 이 중 축소 23열에 남는 것은 5열이다. 그런데 `AI0_Vibration_kurt`·`crest`는 PCA PC2 축을 이루므로(E2) 단독 효과가 작다는 이유로 제외하지 않는다(다변량 재구성에는 기여 가능).
- (b) 상태 간 \|δ\|>0.5 열 16개: AI0 rms·peak·p2p, AI1 rms·peak·p2p·kurt·crest·shape_crest, AI2 rms·peak·p2p·crest, `cur_fit_r2`, `vib_corr01`, `vib_rms_ratio`(축소 23열과의 교집합 13열).
- 정상 q99 대비 이상 중앙값: AI0 rms 1.49배, peak 1.65배, p2p 1.16배 — 진동 진폭만으로는 임계 여유가 작다(21 rule1 fpr/탐지와 일치).
- **모델링 영향**: (1) 진폭 16열은 상태(고/저부하)에 따라 정상 분포 위치가 달라 단일 임계·단일 가우시안이 부적절 → 22·23에 "상태 조건부 정규화(상태별 표준화 또는 상태별 임계)" 변형 행을 추가한다. (2) 이상 구분력은 진폭이 아니라 사인 잔차·전류 형상·진동 상관 방향에 있다 → 진폭 rms 단일 규칙의 한계(21 미탐 3·19·20)와 5번 창의성(사인·관계 피처)의 근거. 판정: (b) **채택**, (a) **참고(보류, ablation 한정)**.

## 4. E6 — 분포·변환

- 가설: rms·peak·crest·kurt는 오른쪽 꼬리가 길어 Gaussian 계열(GMM·Mahalanobis) 가정과 q99 임계가 불안정하고, log 변환이 이를 완화한다(IsolationForest는 단조 변환에 불변).
- 방법: fold 0 train 정상 윈도우 27열의 왜도·초과첨도·Shapiro(표본 5,000 초과 시 무작위 5,000개)를 none/log1p(양수 열은 log도)로 비교. 진폭 rms·peak·crest 9열은 fold별 train 정상 q99의 변동(`stats.quantile_stability`)을 변환 전/후로 비교한다. 로그 공간의 평균은 0에 가까울 수 있어 cv(표준편차/평균)만으로는 비교가 어려우므로 `thr_sd_over_sd`(fold 간 q99 표준편차 / 변환 값의 표준편차)를 함께 둔다.

In [11]:
POS = [c for c in F27 if c.endswith(("_rms", "_peak", "_p2p", "_crest"))]
DS = stats.distribution_summary(TR0, F27, transforms=("none", "log1p"))
DL = stats.distribution_summary(TR0, POS, transforms=("log",))
DS = pd.concat([DS, DL], ignore_index=True); DS["group"] = DS.feature.map(group_of)
DS.to_csv(RES / "distribution_summary.csv", index=False)
print("양수 열(log 대상):", POS)
piv = DS.pivot_table(index="feature", columns="transform", values=["skew", "kurt"]).reindex(F27)
print(piv.round(2).to_string())
SKW = DS.set_index(["feature", "transform"])["skew"]
AMP9 = [f"{ch}_{k}" for ch in ("AI0_Vibration", "AI1_Vibration", "AI2_Current") for k in ("rms", "peak", "crest")]
rows = []
for c in AMP9:
    for tr in ("none", "log"):
        d = W1[["fold", "label"]].copy(); d["_x"] = W1[c] if tr == "none" else np.log(W1[c])
        q = stats.quantile_stability(d, "_x", "fold")
        thr = q.thr.iloc[:-1].astype(float)
        sk, sk0 = SKW.loc[(c, tr)], SKW.loc[(c, "none")]
        sd_tr = float((np.log(TR0[c]) if tr == "log" else TR0[c]).std())
        rows.append({"feature": c, "transform": tr, "cv_thr": float(q.thr.iloc[-1]), "skew_before": sk0, "skew_after": sk,
                     "thr_sd_over_sd": float(thr.std(ddof=1) / sd_tr)})
TE = pd.DataFrame(rows); TE.to_csv(RES / "transform_effect.csv", index=False)
print(TE.round(4).to_string(index=False))
# QQ grid
qc = ["AI0_Vibration_rms", "AI0_Vibration_peak", "AI0_Vibration_crest", "AI0_Vibration_kurt"]
fig, axs = plt.subplots(2, 4, figsize=(15, 7))
for j, c in enumerate(qc):
    x = TR0[c].to_numpy()
    y2 = np.log(x) if x.min() > 0 else (np.log1p(x) if x.min() > -1 else None)
    for i, (tr, y) in enumerate((("변환 전", x), ("log" if x.min() > 0 else "log1p", y2))):
        ax = axs[i, j]
        if y is None:
            ax.text(.5, .5, "log1p 불가 (최소값 %.2f <= -1)" % x.min(), ha="center", transform=ax.transAxes); ax.set_title(c + " " + tr); continue
        sst.probplot(y, dist="norm", plot=ax); ax.get_lines()[0].set_markersize(2)
        ax.set_title("%s %s (skew %.2f)" % (c.replace("AI0_Vibration_", "AI0 "), tr, sst.skew(y))); ax.set_xlabel(""); ax.set_ylabel("")
paths.save_fig(FIG, "qq_grid.png")

양수 열(log 대상): ['AI0_Vibration_rms', 'AI0_Vibration_peak', 'AI0_Vibration_p2p', 'AI0_Vibration_crest', 'AI0_Vibration_shape_crest', 'AI1_Vibration_rms', 'AI1_Vibration_peak', 'AI1_Vibration_p2p', 'AI1_Vibration_crest', 'AI1_Vibration_shape_crest', 'AI2_Current_rms', 'AI2_Current_peak', 'AI2_Current_p2p', 'AI2_Current_crest', 'AI2_Current_shape_crest', 'cur_fit_resid_rms']
                           kurt                  skew              
transform                   log   log1p    none   log  log1p   none
feature                                                            
AI0_Vibration_rms         -0.37   -0.33   -0.24 -0.38   0.41   0.46
AI0_Vibration_peak        -0.41   -0.22   -0.02 -0.35   0.45   0.55
AI0_Vibration_p2p         -0.40   -0.43   -0.21 -0.33   0.40   0.54
AI0_Vibration_kurt          NaN     NaN    1.89   NaN    NaN   1.19
AI0_Vibration_crest       -0.38   -0.30   -0.00  0.19   0.30   0.53
AI0_Vibration_skew          NaN     NaN    0.47   NaN    NaN  -0.07
AI0_Vibration_

saved figures\12_feature_statistics_CHS\qq_grid.png


### E6 판정 (근거 수준 [데이터], fold 0 train 정상 2,605 윈도우)

| 가설 | 판정 | 수치 |
|---|---|---|
| rms·peak·crest가 오른쪽 꼬리가 길다 | **기각** | 진폭 rms·peak·crest 9열 변환 전 왜도 0.36~0.91(최대 `AI1_Vibration_crest` 0.911). 27열 중 \|왜도\|≥1인 열은 5개뿐이며 모두 진폭이 아니다: `cur_fit_f_dev` 26.47(초과첨도 872.9), `cur_ac1` −4.14(33.3), `AI1_Vibration_kurt` 1.85, `vib_rms_ratio` 1.66, `AI0_Vibration_kurt` 1.19 |
| log 변환이 왜도를 \|g\|<1로 낮춘다 | **이미 충족** | log 대상 16열(양수 열) 변환 후 최대 \|왜도\| 0.96(`cur_fit_resid_rms`, 변환 전 −0.40보다 오히려 커짐), \|왜도\|≥1 열 0개. 진폭 9열은 0.36~0.91 → 0.07~0.53(예: AI1 crest 0.911 → 0.533, AI0 rms 0.465 → −0.380) |
| log가 q99 임계 변동을 줄인다 | **채택(효과 작음)** | 변동 계수 cv_thr는 변환 전에도 0.0011~0.0355로 작다(fold 간 q99가 안정). 로그 공간 cv는 평균 부호 때문에 비교 불가라 `thr_sd_over_sd`로 보면 9열 모두 감소 1.1~2.3배: AI0 rms 0.0860 → 0.0408, AI0 peak 0.1841 → 0.0832, AI2 rms 0.0065 → 0.0040, AI0 crest 0.0393 → 0.0293 |
| 정규성 | **기각** | Shapiro p<0.05가 none 27/27열, log 16/16열(표본 2,605·윈도우 종속성으로 정규성 검정은 참고용) |

- kurt·skew·`vib_corr01`·`cur_ac1`은 음수 값이 있어 log가 정의되지 않는다(log1p도 최소값 ≤ −1인 kurt·skew 등은 nan). `log1p`는 rms 스케일(≈0.1)에서 거의 항등이라 효과가 없다.
- **모델링 영향**: log 변환은 필수가 아니라 선택이다. 23(Gaussian 계열) 입력의 rms·peak·p2p(진폭 양수 열)에만 log를 적용하고 임계 q99 변동을 줄이는 이득(1.1~2.3배)을 얻는다. 22 IsolationForest는 단조 변환에 불변이라 무관. 더 큰 위험은 `cur_fit_f_dev`(왜도 26.5)·`cur_ac1`(−4.1)·`vib_rms_ratio`(1.7)의 heavy tail이므로 23 입력에서 순위 변환·클리핑 또는 log(f_dev)·log(1−ac1) 같은 변환을 시험할 후보로 둔다 [추정]. 판정: **보류**(계획서의 "log 표준"은 채택하지 않고 "양수 진폭 열에 선택 적용 + heavy tail 3열 별도 처리"로 정정).

## 5. E7 — 성능 지표 신뢰구간 (21 규칙 재현)

- 가설: 이상 세그먼트 17개로 계산한 AUC 차이(21 rule1 0.786 vs rule3 0.870)는 세그먼트를 재표집하면 구간이 겹친다. 비교표에 점추정만 두면 과대 해석된다.
- 방법: 21 셀 4의 `fit_state`·`rule_scores`와 같은 정의로 `rule1_rms_ai0`·`rule3_rms_state_dir`을 재현한다(출처: `notebooks/21_model_rule_baseline_CHS.ipynb`). kfold OOF 점수를 fold별 train 정상 q99 임계로 정규화(score − thr_fold)해 전체 1초 윈도우에 모은 뒤, 정상·이상 세그먼트를 각각 복원 추출하는 부트스트랩(n_boot=1000)으로 AUC·fpr_segment의 95% 구간과 rule3 − rule1 쌍 차이 구간을 구한다. OOF를 한 번에 합친 점추정 AUC는 21의 fold 평균과 다를 수 있어 21 값을 나란히 둔다.

In [12]:
# 출처: notebooks/21_model_rule_baseline_CHS.ipynb 셀 4 (fit_state / rule_scores 중 rule1·rule3만)
CUR, VIB = "AI2_Current_rms", "AI0_Vibration_rms"
def fit_state(cur_train_normal):
    km = KMeans(n_clusters=2, random_state=42, n_init=10).fit(np.asarray(cur_train_normal, float).reshape(-1, 1))
    lo, hi = np.sort(km.cluster_centers_.ravel())
    return (lo + hi) / 2, (lo, hi)
def rule_scores(w, tr_mask):
    trn = w[tr_mask & (w.label == 0)]
    bound, _ = fit_state(trn[CUR])
    st = (w[CUR] > bound).astype(int).to_numpy(); st_tr = (trn[CUR] > bound).astype(int).to_numpy()
    thr_state = {s: evaluate.threshold_from_normal(trn[VIB].to_numpy()[st_tr == s]) for s in (0, 1)}
    rel = w[VIB].to_numpy() / np.where(st == 1, thr_state[1], thr_state[0])
    return pd.DataFrame({"rule1_rms_ai0": w[VIB].to_numpy(),
                         "rule3_rms_state_dir": rel * (1.0 - w["vib_corr01"].fillna(0.0).to_numpy())}, index=w.index)
RULES = ["rule1_rms_ai0", "rule3_rms_state_dir"]
OOF = pd.DataFrame(index=W1.index, columns=RULES, dtype=float)
for k in range(5):
    tr = (W1.fold != k).to_numpy(); te = np.flatnonzero(W1.fold == k)
    sc = rule_scores(W1, tr)
    for m in RULES:
        thr = evaluate.threshold_from_normal(sc[m][tr & (W1.label == 0).to_numpy()])
        OOF.loc[te, m] = sc[m].iloc[te].to_numpy() - thr
assert not OOF.isna().any().any()
y, g = W1.label.to_numpy(), W1.seg_uid.to_numpy()
rows = []
for m in RULES:
    r = stats.bootstrap_auc_ci(y, OOF[m].to_numpy(), g, n_boot=1000, seed=SEED, thr=0.0)
    a21 = m21.loc[m]
    rows.append({"model": m, "auc": r["auc"], "auc_ci_low": r["auc_ci_low"], "auc_ci_high": r["auc_ci_high"],
                 "fpr_segment": r["fpr_segment"], "fpr_segment_ci_low": r["fpr_segment_ci_low"], "fpr_segment_ci_high": r["fpr_segment_ci_high"],
                 "n_seg_normal": r["n_seg_normal"], "n_seg_outlier": r["n_seg_outlier"], "n_valid": r["n_valid"],
                 "auc_21_foldmean": a21.auc, "fpr_segment_21_foldmean": a21.fpr_segment})
BS = pd.DataFrame(rows); BS.to_csv(RES / "auc_bootstrap_21.csv", index=False)
d = stats.bootstrap_auc_diff(y, OOF[RULES[0]].to_numpy(), OOF[RULES[1]].to_numpy(), g, n_boot=1000, seed=SEED)
DF = pd.DataFrame([{"pair": "rule3_rms_state_dir - rule1_rms_ai0", "diff": d["diff"], "ci_low": d["ci_low"], "ci_high": d["ci_high"],
                    "p_two_sided": d["p_two_sided"], "overlap_zero": d["overlap_zero"], "n_valid": d["n_valid"],
                    "diff_21_foldmean": m21.loc[RULES[1]].auc - m21.loc[RULES[0]].auc}])
DF.to_csv(RES / "auc_diff_21.csv", index=False)
print(BS.round(4).to_string(index=False)); print(DF.round(4).to_string(index=False))
ov = not (BS.auc_ci_low.iloc[1] > BS.auc_ci_high.iloc[0] or BS.auc_ci_low.iloc[0] > BS.auc_ci_high.iloc[1])
print("두 AUC 구간 겹침:", ov, "/ 쌍 차이 구간이 0 포함:", bool(d["overlap_zero"]))

              model    auc  auc_ci_low  auc_ci_high  fpr_segment  fpr_segment_ci_low  fpr_segment_ci_high  n_seg_normal  n_seg_outlier  n_valid  auc_21_foldmean  fpr_segment_21_foldmean
      rule1_rms_ai0 0.7780      0.6410       0.8892       0.0396              0.0245               0.0566           530             17     1000           0.7864                   0.0395
rule3_rms_state_dir 0.8676      0.7371       0.9605       0.0358              0.0208               0.0528           530             17     1000           0.8696                   0.0358
                               pair   diff  ci_low  ci_high  p_two_sided  overlap_zero  n_valid  diff_21_foldmean
rule3_rms_state_dir - rule1_rms_ai0 0.0896   0.042   0.1434        0.001         False     1000            0.0832
두 AUC 구간 겹침: True / 쌍 차이 구간이 0 포함: False


### E7 판정 (근거 수준 [데이터], 정상 530·이상 17 세그먼트 재표집 1,000회 전부 유효)

| 모델 | OOF 합산 AUC [95% CI] | 21 fold 평균 AUC | fpr_segment [95% CI] | 21 fpr_segment |
|---|---|---|---|---|
| rule1_rms_ai0 | 0.7780 [0.6410, 0.8892] | 0.7864 | 0.0396 [0.0245, 0.0566] | 0.0395 |
| rule3_rms_state_dir | 0.8676 [0.7371, 0.9605] | 0.8696 | 0.0358 [0.0208, 0.0528] | 0.0358 |

- 점추정 AUC는 21 fold 평균과 0.002~0.008 차이(OOF 합산 vs fold 평균) — 결론 동일. fpr_segment는 21과 일치.
- 두 AUC의 **개별 95% 구간은 겹친다**(rule1 상한 0.889 > rule3 하한 0.737, 구간 폭 0.248 / 0.223). 그러나 같은 재표집 안의 **쌍 차이(rule3 − rule1) = +0.0896 [0.0420, 0.1434], 양측 p = 0.001(부트스트랩 하한 1/1000), 구간이 0을 포함하지 않는다**(21 fold 평균 차이 0.0832).
- 가설 "차이가 겹쳐 무의미"는 **부분 기각**: 개별 구간 겹침만으로 "차이 없음"이라 하면 틀리고(같은 세그먼트를 공유하는 두 모델의 AUC는 강하게 양의 상관), 쌍 차이 구간으로 판단해야 한다.
- **31 비교표 CI 열 규칙 제안**: (1) 모든 모델 행에 `auc_ci_low/high`, `fpr_segment_ci_low/high`(세그먼트 단위 부트스트랩 n_boot=1000, seed 42)를 붙인다. (2) 두 모델의 우열은 개별 구간 겹침이 아니라 `bootstrap_auc_diff`의 쌍 차이 95% 구간이 0을 포함하는지로 판정하고, 포함하면 "차이 없음", 아니면 "차이 있음"이라 쓴다. (3) p는 1/n_valid가 하한이므로 "p<0.001"로 표기. (4) 이상 세그먼트 17개·이벤트 1건이라는 한계를 표 주석에 항상 병기한다. 판정: **채택**(규칙 (1)~(4)).

## 6. 결론

### 판정 요약

| 절 | 판정 | 핵심 수치 |
|---|---|---|
| E1 상관·중복·VIF | 채택 | 완전 중복 6쌍(형상 kurt·skew) → 27열, 0.9 군집 19개, VIF>10 14열, 조건수 33열 9.7e15 → 27열 99.2 → 축소 23열 30.8 |
| E2 PCA | 채택(PC1=부하 축), 소수 축 가설은 기각 | 90% 10 / 95% 13성분(전 fold 동일), PC1 37.5%, pca_recon 1초 kfold AUC 1.0000·fpr_segment 0.0550·미탐 0, time_block AUC 1.0000·fpr_segment 0.0615 |
| E5 그룹 비교 | (b) 채택, (a) 참고 | 세그먼트 530/17, 226/304, 유의 21/27·25/27, (a) \|δ\|<0.3 6열, (b) \|δ\|>0.5 16열 |
| E6 분포·변환 | 보류 | 진폭 왜도 0.36~0.91(이미 \|g\|<1), log 후 q99 변동 1.1~2.3배 감소, heavy tail은 `cur_fit_f_dev`·`cur_ac1`·`vib_rms_ratio` |
| E7 신뢰구간 | 채택 | rule1 0.778 [0.641, 0.889], rule3 0.868 [0.737, 0.961], 차이 +0.090 [0.042, 0.143] p=0.001 |

### 입력 피처 계약 v2 (초안)

- **열 수**: 33 → 27(완전 중복 6열 제거) → **23(권장 축소 집합)**. 22·23 기본 입력은 축소 23열, 27열은 ablation 비교용.
- **제외 열**: 완전 중복 6열(`AI0/AI1/AI2_Vibration|Current_shape_kurt`·`shape_skew`), 군집 4열(`AI1_Vibration_rms`·`AI1_Vibration_peak`, `AI2_Current_peak`, `AI2_Current_shape_crest`). E5(a) \|δ\|<0.3 열은 라벨 선택이므로 기본 입력에서 제외하지 않고 ablation으로만 쓴다(지도 정보 사용 표기).
- **변환**: 양수 진폭 열(rms·peak·p2p)은 23에서 log 선택 적용, 22는 무관. `cur_fit_f_dev`·`cur_ac1`·`vib_rms_ratio`는 heavy tail이라 순위 변환·클리핑 등 별도 처리 시험(23).
- **상태 정규화 대상**: 상태 간 \|δ\|>0.5 열 중 축소 집합에 남는 13열(`AI0_Vibration_rms`·`peak`·`p2p`, `AI1_Vibration_p2p`·`kurt`·`crest`·`shape_crest`, `AI2_Current_rms`·`p2p`·`crest`, `cur_fit_r2`, `vib_corr01`, `vib_rms_ratio`). 상태는 21 규칙과 같은 정의(전류 AC RMS 2-means 경계)를 쓰며, parquet `state`는 0 = 고부하(전류 AC RMS 중앙값 156.3)·1 = 저부하(86.4)로 규약이 21 셀 내부 상태(1 = 고부하)와 반대이므로 22·23에서 혼용하지 않는다.
- **후보 추가**: 23에 `pca_recon`(성분 13) 행, 22·23에 "상태 조건부 정규화" 변형 행.
- **비교표 규칙**: E7 (1)~(4).

### 확정 페이지 승격 후보 문장 (검정 통과 + CSV 재현, 채택 항목만)

1. "윈도우 피처 33열 중 형상(z-score) 첨도·왜도 6열은 진폭 첨도·왜도와 수치적으로 동일(max\|diff\| ≤ 5.33e-15)하므로 모델 입력에서 제외하며, 33열 그대로는 표준화 공분산 조건수가 9.7e15로 특이하다." (`duplicate_columns.csv`, `vif_table.csv`)
2. "중복 제거 27열의 조건수는 99, 축소 23열은 31이며 VIF>10 열은 14개에서 8개로 준다." (`vif_table*.csv`, `reduced_feature_set.csv`)
3. "정상 PC1(설명 37.5%)은 전류·AI1 진동 진폭 축(부하)이고 95% 분산에는 13성분이 필요하다(전 fold 동일)." (`pca_explained.csv`, `pca_loadings.csv`)
4. "진폭 16열은 고/저부하 사이 Cliff \|δ\|>0.5로 정상 분포 위치가 달라 상태 조건부 정규화 대상이다(세그먼트 226 vs 304)." (`group_test_by_state.csv`)
5. "모델 비교표의 우열은 세그먼트 단위 부트스트랩 쌍 차이 95% 구간이 0을 포함하는지로 판정한다. 예: 21 rule3 − rule1 = +0.090 [0.042, 0.143]." (`auc_bootstrap_21.csv`, `auc_diff_21.csv`)

기각·보류(승격 제외): 소수 축 가설(E2), 진폭군 큰 효과 가설(E5), log 표준화(E6), pca_recon AUC 1.0의 일반화 해석(이벤트 1건·날짜=라벨). 3·4·5번 심사 항목에는 "진폭이 아니라 사인 잔차·전류 crest·진동 상관 방향이 이상을 가른다"는 발견 → 시사점(21 미탐 3·19·20이 진폭 규칙의 한계) → 활용(23 입력 계약·상태 정규화) 흐름으로 연결한다.